# Thesis summary — every number and figure in one place (LOAD-ONLY)

Open this while writing. It **only loads pre-saved results** from `sleep_edf/results/` — it does **not**
recompute attributions, deletion curves, CMI, or anything expensive. **Every cell runs in seconds.** The only
computations are trivial and flagged where they occur: (i) DDS/PES std over the 5 saved per-seed values (CMI
was saved with std, DDS/PES were not); (ii) **one** rank-correlation for Model 2's cross-method agreement,
derived from Model 2's *saved attribution arrays* because its XAI run predates the saved `agreement` key;
(iii) small plots of already-loaded numbers. Nothing regenerates an attribution or a CMI.

Numbers that were produced by one-off diagnostics (the IG investigation, the attention-collapse checks) are
**not** in the results files — they are embedded here as documented constants with a pointer to their
`DECISIONS_LOG.md` entry, not recomputed.

Sections: **§1** ladder (training) · **§2** central CMI/DDS/PES grid · **§3** trend + concentration confound ·
**§4** the method-family split (primary result) · **§5** heatmaps · **§6** attention (separate) ·
**§7** methodology reference · **§8** limitations · **§9** literature.

> Prepared, not executed — run it yourself.

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
from scipy.stats import spearmanr
from IPython.display import Image, display, Markdown

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "sleep_edf").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
MET = PROJECT_ROOT / "sleep_edf" / "results" / "metrics"
FIG = PROJECT_ROOT / "sleep_edf" / "results" / "figures"

def load_json(name):
    p = MET / name
    if not p.exists(): print(f"[MISSING] {name}"); return None
    return json.load(open(p))

def show(name, caption=None):
    p = FIG / name
    if caption: display(Markdown(f"*{caption}*"))
    if p.exists(): display(Image(filename=str(p)))
    else: print(f"[MISSING FIGURE] {name}")

def std_over_seeds(method_entry, key):
    """DDS/PES std from the saved per-seed list (only CMI_std was persisted)."""
    return float(np.std([d[key] for d in method_entry["per_seed"]]))

CLASS_NAMES = ["W", "N1", "N2", "N3", "REM"]
SHARED = ["FeatureAblation", "KernelSHAP", "IntegratedGradients"]
print("results dir:", MET, "| figures dir:", FIG)

## §1. The ladder — models and training

All five rungs (Model 1 = the band-power logistic baseline). Metrics load from the training aggregates;
architecture facts (kernel/patch, receptive field) are documented constants from DECISIONS_LOG / config —
they are not recomputed.

In [ ]:
# Documented architecture facts (from DECISIONS_LOG / sleep_edf/config.py) — not computed here.
LADDER = [
    (1, "model1_bandpower",    "Band-power logistic",          "5 log-band features", "n/a (frequency-domain)"),
    (2, "model2_shallow_cnn",  "Shallow 1D CNN [16,32]",       "kernel 15",           "46 samp / 460 ms"),
    (3, "model3_medium_cnn",   "Medium 1D CNN [32,64,64]",     "kernel 15",           "106 samp / 1060 ms"),
    (4, "model4_deep_cnn",     "Deep 1D CNN [64,128,128,256]", "kernel 15",           "226 samp / 2260 ms"),
    (5, "model5_transformer",  "Transformer (patch 60)",       "patch 60 -> 50 tok",  "global (self-attention)"),
]
hdr = f"{'rung':<5}{'architecture':<30}{'params':>10}{'kernel/patch':<22}{'RF':<24}{'bal.acc':>16}{'macroF1':>16}{'N1 rec':>8}{'N3 rec':>8}{'stop@ep':>9}{'stop':>10}"
print(hdr); print("-"*len(hdr))
for rung, key, arch, kp, rf in LADDER:
    d = load_json(f"{key}_aggregate.json")
    if d is None: continue
    s = d["summary"]; ps = d["per_seed"]; seeds = [str(x) for x in d["seeds"]]; p0 = ps[seeds[0]]
    npar = p0.get("n_params")
    bal = f"{s['balanced_accuracy']['mean']:.4f}±{s['balanced_accuracy']['std']:.3f}"
    mf1 = f"{s['macro_f1']['mean']:.4f}±{s['macro_f1']['std']:.3f}"
    n1 = np.mean([ps[sd]["per_class"]["N1"]["recall"] for sd in seeds])
    n3 = np.mean([ps[sd]["per_class"]["N3"]["recall"] for sd in seeds])
    if "stopped_epoch" in s:
        stopep = f"{s['stopped_epoch']['mean']:.1f}"
        reasons = set(ps[sd].get("stop_reason","?") for sd in seeds); stop = "/".join(sorted(reasons))
    else:
        stopep, stop = "n/a", "n/a (no epochs)"
    print(f"{rung:<5}{arch:<30}{npar:>10,}{kp:<22}{rf:<24}{bal:>16}{mf1:>16}{n1:>8.3f}{n3:>8.3f}{stopep:>9}{stop:>10}")
print("\nN1 and N3 are the load-bearing minority stages. Model 1 (logistic, no epochs) has no early-stopping fields.")

## §2. The central results grid — CMI, DDS, PES (Models 2–5, shared methods)

Model × method, mean ± std over 5 seeds, for the three methods carried across every rung. CMI std is saved;
DDS/PES std are computed from the saved per-seed values (trivial).

**Read FeatureAblation as the ORACLE / CEILING, not a competing method.** With a zero baseline it measures
single-region deletion impact — nearly the operation CMI's deletion curves perform — so it scores high by
construction. **KernelSHAP and Integrated Gradients should be read as "how close to the oracle", not "did they
beat FA".**

In [ ]:
xai = {r: load_json(f"model{r}_xai_cmi_results.json") for r in [2, 3, 4, 5]}
for metric, skey in [("CMI", "CMI"), ("DDS", "DDS"), ("PES", "PES")]:
    print(f"\n=== {metric} (mean ± std over 5 seeds) ===")
    print(f"{'method':<22}" + "".join(f"{'Model '+str(r):>16}" for r in [2,3,4,5]))
    for name in SHARED:
        row = f"{name:<22}"
        for r in [2,3,4,5]:
            me = xai[r]["methods"][name]
            mean = me[f"{skey}_mean"]
            std = me.get(f"{skey}_std") if metric == "CMI" else std_over_seeds(me, skey)
            row += f"{f'{mean:.3f}±{std:.3f}':>16}"
        print(row)
print("\nFeatureAblation = oracle/ceiling. KernelSHAP = the model-agnostic through-line; IG = gradient cross-check.")

## §3. The faithfulness trend and the concentration confound

The saved CMI-vs-rung and CMI-vs-concentration plot (all four rungs), plus the concentration table.

**Reading:** concentration rose 0.087 → 0.115 → 0.131 then **FELL to 0.090** at Model 5, while FA and KS CMI
**jumped +0.170 and +0.131** there. The mechanical explanation — CMI rises because reliance concentrates,
making deletion curves separate more easily — **cannot account for the largest step** (concentration went the
wrong way at Model 5). The confound control clears the perturbation-based trend.

In [ ]:
print(f"{'rung':<10}{'concentration (mean±std)':>26}{'FA CMI':>10}{'KS CMI':>10}{'IG CMI':>10}")
prev_c = None
for r in [2, 3, 4, 5]:
    c = xai[r]["concentration"]; m = xai[r]["methods"]
    cstr = f"{c['mean']:.3f}±{c['std']:.3f}"
    delta = f"  (Δ {c['mean']-prev_c:+.3f})" if prev_c is not None else ""
    print(f"Model {r:<4}{cstr:>26}{m['FeatureAblation']['CMI_mean']:>10.3f}"
          f"{m['KernelSHAP']['CMI_mean']:>10.3f}{m['IntegratedGradients']['CMI_mean']:>10.3f}{delta}")
    prev_c = c["mean"]
print("\nConcentration 0.087 -> 0.115 -> 0.131 -> 0.090 (FELL at M5); FA/KS CMI jumped at M5 -> not mechanical.")
show("sleep_edf_13_model5_xai_cmi_vs_concentration.png", "CMI & concentration across the ladder (Models 2-5)")

## §4. THE METHOD-FAMILY SPLIT — a primary result

**Perturbation-based faithfulness rises across the ladder; gradient-based faithfulness is approximately
INVARIANT.** State it exactly this way — **do not** write "faithfulness rises on every shared method": IG is a
shared method and it does not rise.

- FA CMI: 0.428 / 0.475 / 0.484 / **0.654**   ·   KS CMI: 0.389 / 0.457 / 0.479 / **0.610**
- IG CMI: 0.387 / 0.427 / 0.427 / **0.432**   (and **0.402** on the patch-15 transformer)
- Families agree closely on the CNNs (FA–IG ~0.70) and **diverge sharply at the transformer (0.275)**.

Cross-method rank agreement is loaded from each rung's saved results; **Model 2's is derived from its saved
attribution arrays** (its XAI run predates the saved `agreement` key) — a rank correlation, seconds, no
attribution recompute.

In [ ]:
# Cross-method rank agreement per rung. Models 3-5 have it saved; Model 2 derived from saved attributions.
def model2_agreement():
    z = np.load(MET / "model2_xai_fa_ig_attr.npz")
    fa, ig = z["fa_attr"], z["ig_attr"]
    ks = np.stack([np.load(MET / f"model2_xai_ks_attr_seed{s}.npy") for s in range(5)])
    def pa(A, B):
        v = [spearmanr(A[si, j], B[si, j]).correlation for si in range(A.shape[0]) for j in range(A.shape[1])]
        v = np.array(v, float); return float(np.nanmean(v))
    return {"FA-KS": pa(fa, ks), "FA-IG": pa(fa, ig), "KS-IG": pa(ks, ig)}

agree = {}
for r in [2, 3, 4, 5]:
    a = xai[r].get("agreement")
    if a is None and r == 2:
        a = model2_agreement(); print("Model 2 agreement DERIVED from saved attributions (json predated the key).")
    agree[r] = {k: a.get(k) for k in ["FA-KS", "FA-IG", "KS-IG"]}   # shared methods only (drop attention pairs)

print(f"\n{'pair':<8}" + "".join(f"{'Model '+str(r):>12}" for r in [2,3,4,5]))
for pair in ["FA-KS", "FA-IG", "KS-IG"]:
    print(f"{pair:<8}" + "".join(f"{agree[r][pair]:>12.3f}" for r in [2,3,4,5]))
print("\nFA-IG collapses ~0.70 (CNNs) -> 0.275 (transformer): the perturbation vs gradient families split.")

In [ ]:
# Family-split plot from the loaded numbers (no recomputation).
import matplotlib.pyplot as plt
rungs = [2, 3, 4, 5]
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
for name, mk in [("FeatureAblation","o-"),("KernelSHAP","s-"),("IntegratedGradients","^-")]:
    ax[0].plot(rungs, [xai[r]["methods"][name]["CMI_mean"] for r in rungs], mk, label=name)
ax[0].set_xticks(rungs); ax[0].set_xlabel("rung"); ax[0].set_ylabel("CMI"); ax[0].legend(fontsize=8)
ax[0].set_title("CMI by method — FA/KS rise, IG ~invariant")
for pair, mk in [("FA-KS","o-"),("FA-IG","s-"),("KS-IG","^-")]:
    ax[1].plot(rungs, [agree[r][pair] for r in rungs], mk, label=pair)
ax[1].set_xticks(rungs); ax[1].set_xlabel("rung"); ax[1].set_ylabel("rank agreement (Spearman)")
ax[1].set_title("Cross-method agreement — collapses at the transformer"); ax[1].legend(fontsize=8)
fig.suptitle("§4 The perturbation-vs-gradient method-family split", y=1.02); fig.tight_layout(); plt.show()

### §4 — the refuted hypothesis (IG investigation), for the write-up

The attention-collapse → smeared-gradients mechanism was tested and **refuted** (DECISIONS_LOG: *Investigation:
Model 5's IG anomaly*). These are documented diagnostic constants, not recomputed here:
- Per-layer gradient magnitude is **uniform ~4e-05 across all six encoder layers** — structured layers (0, 2)
  and collapsed layers (1, 3, 4, 5) indistinguishable; gradients do **not** collapse where attention does.
- IG's per-region coefficient of variation is **1.374 vs FA's 0.885** — IG is *more* structured than FA, not
  smeared. It is ordered differently, not flattened.
- Per-sample correlation between attention collapse and IG–FA disagreement is **0.078** — essentially zero.
- The effect **travels to patch-15** (FA 0.559, IG 0.402, FA–IG 0.332), a different transformer.
- IG completeness residual is **0.0003** (transformer) / 0.0008 (CNN) — the zero baseline is not implicated.

Conclusion to write: the split is a **perturbation-vs-gradient family difference** (FA/KS reward deletion
impact = what CMI measures; IG measures path-integrated gradients); *why* it widens specifically at the
transformer is **not** mechanistically established.

## §5. Attribution heatmaps (shared methods)

Saved per-region attribution heatmaps (per predicted class × region over the 30-s epoch, mean over seeds).
Models 2–4 show FA/KS/IG. **Model 5's saved heatmaps bundle attention** (no shared-methods-only version was
saved) — its heatmaps are shown in §6 to keep attention out of this section.

In [ ]:
show("sleep_edf_10_model2_xai_heatmaps_all.png", "Model 2 (shallow CNN) — FA / KS / IG")
show("sleep_edf_11_model3_xai_heatmaps_all.png", "Model 3 (medium CNN) — FA / KS / IG")
show("sleep_edf_11_model4_xai_heatmaps_all.png", "Model 4 (deep CNN) — FA / KS / IG")
print("Model 5 heatmaps include attention -> shown in §6.")
print("Note: Models 2-3 were near-uniform within the epoch (GAP head discards temporal position);")
print("class-level structure appears (N1/N3 vs REM) but not within-class temporal localisation.")

## §6. ATTENTION — separate, not comparable to the other three

**Kept out of §2–§5 deliberately.** FA, KS and IG measure attribution quality; attention's score measures the
**near-absence of any attribution**. It is not a fourth point on the same faithfulness scale.

**Two treatments, both pre-registered before results were seen** (DECISIONS_LOG: *Model 5 attention*):
- **Last-layer (PRIMARY): a DEGENERATE failure.** Attention rank collapse — last-layer attention is exactly
  uniform (**1/50, std 0.000000 across all 8 heads**), **input-independent** (bit-identical for a predicted-W
  vs predicted-N3 sample), with only layers 0 and 2 non-uniform (std 0.0158 / 0.0084). There is **no structure
  to assess**, so this is **NOT** the Jain & Wallace case — it is the degenerate limiting case.
- **Rollout (SECONDARY): the Jain & Wallace case, properly demonstrated.** Non-degenerate (incorporates the
  structured layers 0/2), so it has real input-dependent structure — but that structure is **essentially
  uncorrelated with feature importance** (rank agreement 0.023 with FA, 0.034 with IG). Attention that HAS
  structure but fails to track importance = Jain & Wallace (2019), reproduced in a time-series setting.

**Bound:** this transformer **underfit** (0.6603, worst rung), so the claim is about *this model*; whether a
well-fit transformer would fail is untested (future work). Numbers below load from the saved XAI/rollout
results; the collapse diagnostics are documented constants from the DECISIONS_LOG entry.

In [ ]:
att = xai[5]["methods"]["Attention"]
roll = load_json("model5_xai_rollout_results.json")
print("LAST-LAYER attention (PRIMARY, degenerate):")
print(f"  CMI {att['CMI_mean']:.3f} ± {att['CMI_std']:.3f} | DDS {att['DDS_mean']:.3f} | PES {att['PES_mean']:.3f}")
print("  rank agreement vs FA/KS/IG: UNDEFINED (constant 1/50 vector -> Spearman undefined)")
print("  collapse diagnostics (documented): exactly 1/50, std 0.000000 across all 8 heads; bit-identical")
print("  for predicted-W vs predicted-N3; layers 0 & 2 non-uniform (std 0.0158 / 0.0084).")
print("\nROLLOUT (SECONDARY, the Jain & Wallace case):")
if roll:
    print(f"  CMI {roll['CMI_mean']:.3f} ± {roll['CMI_std']:.3f} | DDS {roll['DDS_mean']:.3f} | PES {roll['PES_mean']:.3f}")
    print(f"  rank agreement: FA {roll['agreement_with_FA']:.3f} | IG {roll['agreement_with_IG']:.3f}  (~zero)")
    print(f"  per-region std across regions: {roll['roll_std_across_regions']:.4f}  (>0 -> non-degenerate)")
show("sleep_edf_13_model5_xai_heatmaps_cheap.png", "Model 5 — FA / IG / Attention (attention panel is ~uniform)")
show("sleep_edf_13_model5_xai_rollout_heatmap.png", "Model 5 — attention ROLLOUT (has structure, uncorrelated with importance)")

## §7. Methodology reference — locked decisions

Each with a one-line justification and its `DECISIONS_LOG.md` entry (see the log for the full reasoning trail).

| decision | value | DECISIONS_LOG entry |
|---|---|---|
| Training subsample | 20,000 epochs, stratified, seed 42 | *SUBSAMPLING* |
| Fixed split | 17,742 train / 2,258 val (subject-level, seed 8) / 40,145 test | *subject-level early-stopping validation split* |
| XAI eval subset | 500, stratified, seed 42 (fixed ladder-wide) | *Model 2 XAI/CMI run* |
| KernelSHAP n_samples | 8,000 (does not fully converge; abs. CMI biased low, comparable within-study only) | *XAI/CMI phase parameters* |
| Perturbation method | `zero` (uniform hiding; stage-neutral on z-scored EEG) | *XAI/CMI phase parameters* |
| Target class | PREDICTED (faithfulness = explain what the model computed) | *Target class = PREDICTED* |
| Device split | FA on CPU, IG/KS(+batched)/attention on MPS (measured) | *XAI/CMI phase parameters* / *MPS device* |
| Seeds | 5, per-seed then mean ± std | training entries |
| Stopping rule | monitor val balanced-accuracy, patience 10, min_delta 0.002, max_epochs 100 | *ladder-wide early-stopping protocol* |
| KS coalition batching | `perturbations_per_eval=200` (equivalence-gated 8e-8) | *perturbations_per_eval on kernel_shap* |
| Attention reduction | last-layer, mean-over-heads, attention-received (rollout = pre-registered follow-up) | *Model 5 attention* |
| Region grid | 50 regions of 60 samples (600 ms; one token per region at patch 60) | *Region-size* / *Transformer patch size = 60* |

## §8. Limitations and open questions

1. **The headline cannot distinguish "faithfulness rises" from "deletion-measured faithfulness rises."** Two
   of the three shared methods are aligned with what CMI measures — **FA is near-circular with it by
   construction** (single-region deletion impact vs cumulative deletion in FA's own order) and **KS rewards
   the same deletion impact** — while the one method measuring a genuinely different quantity (**IG**,
   path-integrated gradients) **does not rise**. State the trend as strongest for the deletion-based methods.
2. **Why the perturbation-vs-gradient split widens specifically at the transformer is NOT established.** The
   attention-collapse hypothesis was tested and refuted (§4); no causal mechanism is claimed.
3. **Fit-quality confound at Model 5.** Model 5 has the highest perturbation-based faithfulness and the *worst*
   accuracy (0.6603) — "explanations of a poorly-fit model may be easier to be faithful to." **Counter-
   evidence:** Models 3 and 4 have near-identical accuracy (0.7453 / 0.7439) and near-identical CMI, and Model
   2 has higher accuracy than Model 5 with much lower CMI. So fit quality does not cleanly explain the trend.
4. **PES saturation** makes CMI effectively DDS-only on the CNNs (PES pinned ~1.0), so the consistency
   component does not discriminate there.
5. **KernelSHAP does not converge** over a 50-region grid within tractable compute; absolute CMI is biased low
   and comparable **within this study only**, not against published CMI figures.
6. **N = 500 stability** was verified on one model only (Model 2 bootstrap).
7. **The 4→5 step is an architecture-family change, not a parameter step** (~1.4× params vs ~10× elsewhere);
   the **3→4 step is the controlled comparison** (9× params, near-flat accuracy).
8. **Depth/width are confounded** across the CNN rungs (varied together); parameter count is the declared axis.
9. **Attention is unmeasurable by any faithfulness metric here** (degenerate last-layer; uncorrelated rollout);
   named as future work.

## §9. Literature to cite (reference list — no computation)

Two papers found *after* the experiments; to be cited, not implemented:

- **Mehrpanah, Achlioptas et al. (2025), ICCV — "On the Complexity-Faithfulness Trade-off of Gradient-Based
  Explanations."** Grounds the metric-family concern (existing metrics are "influenced by extraneous factors
  such as the choice of baseline and removal order") and offers a candidate mechanism (perturbation-based
  methods act as low-pass filters). **Cannot be implemented here:** its framework scores only *gradient-based*
  methods — KernelSHAP and FeatureAblation appear nowhere in it.
- **Yeh, Hsieh et al. (2019), NeurIPS — "On the (In)fidelity and Sensitivity of Explanations."** A
  non-deletion *objective* metric; recorded as future work. **Note it would NOT resolve the family-alignment
  problem:** its Proposition 2.5 proves Shapley values are optimal for infidelity under a specific
  perturbation distribution — so it too has a built-in alignment with a perturbation-based method.

---

## §10. Random-attribution floor (scale calibration)

The CMI floor from random attribution vectors pushed through the same deletion+CMI machinery (see
`notebooks/baseline/00_random_baseline.ipynb`; random seed logged in DECISIONS_LOG). It calibrates the scale:
how much of each method's CMI is signal above chance ordering. Loads `random_floor_results.json` — **run the
baseline notebook first**; if absent, this section says so and skips. The floor is added as a horizontal
reference line on a CMI-vs-rung plot here (the §4 plot is left unchanged).

In [ ]:
rf_path = MET / "random_floor_results.json"
if not rf_path.exists():
    print("[random floor not computed] run notebooks/baseline/00_random_baseline.ipynb first.")
else:
    rf = json.load(open(rf_path)); fl = rf["floor"]
    print(f"random-vector seed: {rf['random_seed']} | N_eval {rf['n_eval']} | PM {rf['pm']}")
    print(f"\n{'model':<9}{'RANDOM floor CMI':>18}{'FA':>9}{'KS':>9}{'IG':>9}   (margins above floor)")
    for r in [2,3,4,5]:
        f0 = fl[str(r)]["CMI_mean"]; m = xai[r]["methods"]
        fa,ks,ig = m['FeatureAblation']['CMI_mean'], m['KernelSHAP']['CMI_mean'], m['IntegratedGradients']['CMI_mean']
        print(f"Model {r:<3}{f0:>18.3f}{fa:>9.3f}{ks:>9.3f}{ig:>9.3f}   (+{fa-f0:.3f} / +{ks-f0:.3f} / +{ig-f0:.3f})")
    fl5 = fl["5"]["CMI_mean"]; att = xai[5]["methods"]["Attention"]["CMI_mean"]
    roll = load_json("model5_xai_rollout_results.json")["CMI_mean"]
    print(f"\n[SEPARATE — attention, not comparable] Model 5 floor {fl5:.3f}: last-layer {att:.3f} (margin {att-fl5:+.3f}) | rollout {roll:.3f} ({roll-fl5:+.3f})")

    import matplotlib.pyplot as plt
    rungs=[2,3,4,5]; mean_floor=float(np.mean([fl[str(r)]["CMI_mean"] for r in rungs]))
    fig,ax=plt.subplots(figsize=(7,4.2))
    for name,mk in [("FeatureAblation","o-"),("KernelSHAP","s-"),("IntegratedGradients","^-")]:
        ax.plot(rungs,[xai[r]["methods"][name]["CMI_mean"] for r in rungs],mk,label=name)
    ax.axhline(mean_floor,color="grey",ls="--",label=f"random floor (mean {mean_floor:.3f})")
    ax.plot(rungs,[fl[str(r)]["CMI_mean"] for r in rungs],"x",color="grey",alpha=.6,label="floor per rung")
    ax.set_xticks(rungs); ax.set_xlabel("rung"); ax.set_ylabel("CMI"); ax.legend(fontsize=8)
    ax.set_title("CMI vs rung with the random floor"); fig.tight_layout(); plt.show()